In [1]:
# ======================================================================
# 0. Environment Setup & Installations (Colab / Jupyter)
# ======================================================================
!git clone https://github.com/zengxianyu/sketchedit.git || true
!pip install -q diffusers accelerate controlnet_aux transformers gradio Pillow xformers

import sys
if '/content/sketchedit' not in sys.path:
    sys.path.append('/content/sketchedit')

# Navigate, grant permissions, and download models for SketchEdit
%cd /content/sketchedit
!chmod +x download/* test_places.sh
!./download/download_model.sh
%cd /content

import os
import glob
import subprocess
import zipfile
import re
from pathlib import Path
from datetime import datetime

import torch
import numpy as np
from PIL import Image, ImageDraw, ImageOps, ImageFilter
import gradio as gr
from diffusers import (
    StableDiffusionControlNetPipeline,
    StableDiffusionControlNetInpaintPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
)

# ======================================================================
# 1. Model Setup (ControlNet Scribble + Stable Diffusion Inpaint)
# ======================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Booting Base Generation Models on {device}...")

try:
    controlnet_model = ControlNetModel.from_pretrained(
        "lllyasviel/control_v11p_sd15_scribble",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    )

    controlnet_pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet_model,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        safety_checker=None,
    )
    controlnet_pipe.scheduler = UniPCMultistepScheduler.from_config(controlnet_pipe.scheduler.config)

    # --- NEW: Load Stable Diffusion Inpainting Pipeline ---
    inpaint_pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
        "runwayml/stable-diffusion-inpainting",
        controlnet=controlnet_model,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        safety_checker=None,
    )
    inpaint_pipe.scheduler = UniPCMultistepScheduler.from_config(inpaint_pipe.scheduler.config)

    if device == "cuda":
        controlnet_pipe = controlnet_pipe.to(device)
        controlnet_pipe.enable_xformers_memory_efficient_attention()

        inpaint_pipe = inpaint_pipe.to(device)
        inpaint_pipe.enable_xformers_memory_efficient_attention()

except Exception as e:
    print(f"ControlNet Error: {e}")

# ======================================================================
# 2. Session Gallery Utilities
# ======================================================================
GALLERY_DIR = Path('/content/session_gallery')
GALLERY_DIR.mkdir(exist_ok=True)
_gallery_images = []

def save_to_gallery(img: Image.Image, label: str = '') -> str:
    """Saves generated images and updates the session history."""
    ts = datetime.now().strftime('%H%M%S_%f')[:10]
    name = f'{ts}_{re.sub(r"[^a-zA-Z0-9]", "_", label)[:20]}.png'
    path = GALLERY_DIR / name
    img.save(path)
    _gallery_images.append(str(path))
    return str(path)

def get_gallery_images():
    """Retrieves all session images for the UI Gallery."""
    results = []
    for p in _gallery_images:
        try:
            results.append((Image.open(p), Path(p).stem))
        except Exception:
            pass
    return results

def download_gallery_zip():
    """Zips the gallery directory for download."""
    zip_path = '/content/session_gallery.zip'
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for p in _gallery_images:
            if Path(p).exists():
                zf.write(p, Path(p).name)
    return zip_path

# ======================================================================
# 3. SketchEdit Wrapper (Native Subprocess Execution)
# ======================================================================
class SketchEditPipeline:
    def __init__(self):
        self.base_temp = '/content/sketchedit/temp_data'

        os.makedirs(os.path.join(self.base_temp, 'images'), exist_ok=True)
        os.makedirs(os.path.join(self.base_temp, 'sketches'), exist_ok=True)

        self.list_file = os.path.join(self.base_temp, 'image_list.txt')
        with open(self.list_file, 'w') as f:
            f.write('input\n')

        print("SketchEdit Subprocess Pipeline is ONLINE.")

    def edit(self, base_image_pil, partial_sketch_pil):
        img_path = os.path.join(self.base_temp, 'images', 'input.png')
        skt_path = os.path.join(self.base_temp, 'sketches', 'input.png')

        base_image_pil.convert('RGB').resize((256, 256), Image.LANCZOS).save(img_path)
        partial_sketch_pil.convert('L').resize((256, 256), Image.NEAREST).save(skt_path)

        out_dir = os.path.join(self.base_temp, 'outputs')
        if os.path.exists(out_dir):
            import shutil
            shutil.rmtree(out_dir)
        os.makedirs(out_dir, exist_ok=True)

        cmd = [
            sys.executable, "test.py",
            "--name", "places",
            "--model", "editline2",
            "--netG", "deepfillc2",
            "--dataset_mode", "testimage",
            "--image_dirs", os.path.join(self.base_temp, 'images'),
            "--mask_dirs", os.path.join(self.base_temp, 'sketches'),
            "--image_lists", self.list_file,
            "--output_dir", out_dir,
            "--checkpoints_dir", "/content/sketchedit/checkpoints",
            "--image_postfix", ".png",
            "--mask_postfix", ".png",
            "--gpu_ids", "0" if torch.cuda.is_available() else "-1",
            "--pool_type", "max",
            "--batchSize", "1",
            "--nThreads", "0",
            "--joint_train_inp",
            "--preprocess", "resize_and_crop",
            "--load_size", "256",
            "--crop_size", "256"
        ]

        result = subprocess.run(cmd, cwd="/content/sketchedit", capture_output=True, text=True)

        if result.returncode != 0:
            raise RuntimeError(f"Subprocess Failed. Error Log:\n{result.stderr[-1000:]}")

        out_files = glob.glob(os.path.join(out_dir, '**', '*.*'), recursive=True)
        out_files = [f for f in out_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        if not out_files:
            raise RuntimeError(f"Model ran but no image was saved.\nSTDOUT:\n{result.stdout[-1000:]}\nSTDERR:\n{result.stderr[-1000:]}")

        synthesized_files = [f for f in out_files if 'synthesized' in f.lower() or 'comp' in f.lower()]
        latest_out = synthesized_files[0] if synthesized_files else sorted(out_files, key=os.path.getmtime)[-1]

        out_pil = Image.open(latest_out).convert('RGB')
        return out_pil.resize((512, 512), Image.LANCZOS)

sketch_edit_pipe = SketchEditPipeline()

# ======================================================================
# 4. Extraction Utilities
# ======================================================================
def extract_drawing_controlnet(data):
    if data is None or not isinstance(data, dict):
        return None, None
    bg_raw = data.get("background")
    layers = data.get("layers", [])
    if bg_raw is None:
        bg = Image.new("RGB", (512, 512), (255, 255, 255))
    else:
        if bg_raw.mode in ('RGBA', 'LA') or (bg_raw.mode == 'P' and 'transparency' in bg_raw.info):
            bg = Image.new("RGB", bg_raw.size, (255, 255, 255))
            bg.paste(bg_raw, mask=bg_raw.split()[-1])
        else:
            bg = bg_raw.convert("RGB")
    has_drawing = False
    stroke = Image.new("RGBA", bg.size, (0, 0, 0, 0))
    for layer in layers:
        if layer is not None:
            stroke.alpha_composite(layer.convert("RGBA"))
            if stroke.split()[3].getextrema()[1] > 0:
                has_drawing = True
    sketch = Image.new("L", bg.size, 0)
    if has_drawing:
        alpha = stroke.split()[3]
        white_stroke = Image.new("L", bg.size, 255)
        sketch.paste(white_stroke, mask=alpha)
    else:
        gray = bg.convert("L")
        sketch = ImageOps.invert(gray)
        sketch = sketch.point(lambda p: 255 if p > 80 else 0)
    return bg, sketch

def extract_sketchedit_mask(data):
    if data is None or not isinstance(data, dict):
        return None, None
    bg_raw = data.get("background")
    layers = data.get("layers", [])
    if bg_raw is None:
        return None, None
    bg = bg_raw.convert("RGB")
    stroke = Image.new("RGBA", bg.size, (0,0,0,0))
    for layer in layers:
        if layer is not None:
            stroke.alpha_composite(layer.convert("RGBA"))
    sketch = Image.new("L", bg.size, 0)
    alpha = stroke.split()[3]
    if alpha.getextrema()[1] > 0:
        white_stroke = Image.new("L", bg.size, 255)
        sketch.paste(white_stroke, mask=alpha)
    return bg, sketch

def extract_inpaint_mask(data):
    """Extraction for Inpainting (creates a dilated mask over brush strokes)"""
    if data is None or not isinstance(data, dict):
        return None, None
    bg_raw = data.get("background")
    layers = data.get("layers", [])
    if bg_raw is None:
        return None, None
    bg = bg_raw.convert("RGB")
    stroke = Image.new("RGBA", bg.size, (0, 0, 0, 0))
    has_drawing = False
    for layer in layers:
        if layer is not None:
            stroke.alpha_composite(layer.convert("RGBA"))
            if stroke.split()[3].getextrema()[1] > 0:
                has_drawing = True

    mask = Image.new("L", bg.size, 0)
    if has_drawing:
        alpha = stroke.split()[3]
        mask.paste(Image.new("L", bg.size, 255), mask=alpha)
        # Dilate mask slightly to cover edges smoothly
        mask = mask.filter(ImageFilter.MaxFilter(11))
    return bg, mask

def make_text_image(msg: str, size=(512, 512)):
    img = Image.new("RGB", size, color=(30, 41, 59))
    draw = ImageDraw.Draw(img)
    draw.text((20, 256), msg, fill=(255, 100, 100))
    return img

# ======================================================================
# 5. Core Logic Wrappers
# ======================================================================
def run_controlnet(gr_input, prompt):
    bg, sketch = extract_drawing_controlnet(gr_input)
    if bg is None:
        err = make_text_image("Error: Canvas is empty.")
        return err, err

    scribble_mask = sketch.convert("RGB").resize((512, 512), Image.LANCZOS)
    try:
        img = controlnet_pipe(
            prompt=prompt,
            negative_prompt="low quality, messy, faces, ugly, bad proportions, distorted",
            image=scribble_mask,
            num_inference_steps=30,
            guidance_scale=7.5
        ).images[0]

        save_to_gallery(img, label="generation") # Save to gallery
    except Exception as e:
        err = make_text_image(f"Generation Error:\n{str(e)}")
        return err, scribble_mask

    return img, scribble_mask

def run_sketchedit_process(gr_input):
    bg, sketch = extract_sketchedit_mask(gr_input)
    if bg is None:
        return make_text_image("Error: Upload an image to edit.")
    try:
        out_img = sketch_edit_pipe.edit(bg, sketch)
        save_to_gallery(out_img, label="sketchedit") # Save to gallery
        return out_img
    except Exception as e:
        return make_text_image(f"SketchEdit Error:\n{str(e)}")

def run_inpaint_process(gr_input, prompt):
    bg, mask = extract_inpaint_mask(gr_input)
    if bg is None:
        return make_text_image("Error: Upload an image to inpaint.")
    if mask is None or mask.getextrema()[1] == 0:
        return make_text_image("Error: Draw the region to inpaint.")

    init_img = bg.resize((512, 512), Image.LANCZOS)
    mask_r = mask.resize((512, 512), Image.NEAREST)
    control_hint = mask_r.convert("RGB") # Use the stroke structure as a controlnet hint

    try:
        out_img = inpaint_pipe(
            prompt=prompt,
            image=init_img,
            mask_image=mask_r,
            control_image=control_hint,
            num_inference_steps=30,
            guidance_scale=7.5
        ).images[0]

        save_to_gallery(out_img, label="inpaint") # Save to gallery
        return out_img
    except Exception as e:
        return make_text_image(f"Inpaint Error:\n{str(e)}")

# ======================================================================
# 6. Professional Gradio UI Layout (Dark Theme + Status Indicator)
# ======================================================================
custom_css = """
footer {visibility: hidden !important;}
/* 1. Black/Dark Theme Backgrounds */
.gradio-container {font-family: 'Inter', system-ui, sans-serif !important; background-color: #0f172a !important; color: #e2e8f0 !important;}
/* 2. Smaller Nav Bar */
#header-container {text-align: center; margin-bottom: 15px; padding: 15px 20px; background: linear-gradient(135deg, #000000, #1e1b4b); color: white; border-radius: 12px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.5);}
#header-container h1 {font-weight: 800; font-size: 2em; margin-bottom: 5px; color: white !important;}
#header-container p {font-size: 1em; color: #a5b4fc; opacity: 0.9; margin-top: 0;}
/* Dark Mode Panel Cards */
.panel-card {border: 1px solid #334155 !important; border-radius: 12px; background: #1e293b !important; padding: 20px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.3);}
.section-title {font-weight: 700; color: #f8fafc !important; font-size: 1.2em; margin-bottom: 15px; border-bottom: 2px solid #334155; padding-bottom: 10px;}
/* Status Text Styling */
.status-text {text-align: center; font-weight: 600; color: #fbbf24 !important; margin-top: 10px;}
"""

with gr.Blocks(theme=gr.themes.Base(), css=custom_css) as demo:

    # --- HEADER ---
    gr.HTML("""
    <div id="header-container">
        <h1>✨ SketchControl-AI Studio</h1>
        <p>Professional Sketch-to-Image Synthesis & Editing</p>
    </div>
    """)

    # ==========================================
    # SECTION 1: ControlNet Generation
    # ==========================================
    with gr.Row():
        with gr.Column(scale=2, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>🖍️ 1. Studio Canvas (Generation)</div>")
            raw_sketch_input = gr.ImageEditor(
              label="Upload a Sketch OR Draw Here", type="pil", interactive=True, height=450,
              brush=gr.Brush(colors=["#000000"], color_mode="fixed", default_size=4)
            )

        with gr.Column(scale=1, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>📝 2. Art Direction</div>")
            prompt_input = gr.Textbox(
                label="Scene Prompt",
                placeholder="E.g., A highly detailed futuristic city, cinematic lighting, 8k resolution...",
                value="A highly detailed landscape masterpiece, lush mountains, clear lake, cinematic lighting",
                lines=5
            )
            gr.Markdown("<br>")
            gen_btn = gr.Button("🚀 Generate Render", variant="primary", size="lg")
            status_indicator = gr.Markdown("", elem_classes="status-text")

    # Output Analytics
    gr.HTML("<br><div class='section-title' style='text-align:center; background:#1e293b; padding:15px; border-radius:12px; border: 1px solid #334155; color: white;'>📊 Phase 1: Output Analytics</div>")

    with gr.Row():
        with gr.Column(scale=1, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>🔍 AI Structure Vision</div>")
            mask_output = gr.Image(label="", type="pil", height=450, interactive=False)

        with gr.Column(scale=1, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>🖼️ Final Render</div>")
            base_output = gr.Image(label="", type="pil", height=380, interactive=False, show_download_button=True)

            with gr.Row():
                send_to_edit_btn = gr.Button("⬇️ Route to SketchEdit", variant="secondary")
                send_to_inpaint_btn = gr.Button("⬇️ Route to Inpaint", variant="secondary")

    # ==========================================
    # SECTION 2: SketchEdit Refining
    # ==========================================
    gr.HTML("<br><div class='section-title' style='text-align:center; background:#1e293b; padding:15px; border-radius:12px; border: 1px solid #334155; color: white;'>🛠️ Phase 2: SketchEdit Refinement</div>")

    with gr.Row():
        with gr.Column(scale=2, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>🖌️ SketchEdit Canvas</div>")
            sketchedit_input = gr.ImageEditor(
              label="Edit Mask Upload / Canvas", type="pil", interactive=True, height=450,
              brush=gr.Brush(colors=["#000000"], color_mode="fixed", default_size=4)
            )

        with gr.Column(scale=1, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>⚙️ Edit Action</div>")
            edit_btn = gr.Button("🎨 Apply SketchEdit", variant="primary", size="lg")
            edit_status = gr.Markdown("", elem_classes="status-text")
            gr.HTML("<div class='section-title' style='margin-top:20px;'>✨ Refined Result</div>")
            sketchedit_output = gr.Image(label="", type="pil", height=280, interactive=False, show_download_button=True)

    # ==========================================
    # SECTION 3: Inpainting (NEW)
    # ==========================================
    gr.HTML("<br><div class='section-title' style='text-align:center; background:#1e293b; padding:15px; border-radius:12px; border: 1px solid #334155; color: white;'>🪄 Phase 3: SD Inpainting (Mask & Edit)</div>")

    with gr.Row():
        with gr.Column(scale=2, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>🖍️ Inpaint Canvas</div>")
            gr.Markdown("<span style='color: #94a3b8; font-size: 0.9em;'>Draw a mask over the object you want to alter or remove.</span>")
            inpaint_input = gr.ImageEditor(
              label="Inpaint Canvas", type="pil", interactive=True, height=450,
              brush=gr.Brush(colors=["#ffffff"], color_mode="fixed", default_size=20)
            )

        with gr.Column(scale=1, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>⚙️ Inpaint Action</div>")
            inpaint_prompt = gr.Textbox(
                label="What to synthesize in the masked area?",
                placeholder="E.g., A glowing red crystal, clean background...",
                lines=3
            )
            gr.Markdown("<br>")
            inpaint_btn = gr.Button("✨ Apply Inpaint", variant="primary", size="lg")
            inpaint_status = gr.Markdown("", elem_classes="status-text")
            gr.HTML("<div class='section-title' style='margin-top:20px;'>✨ Inpaint Result</div>")
            inpaint_output = gr.Image(label="", type="pil", height=230, interactive=False, show_download_button=True)

    # ==========================================
    # SECTION 4: Session Gallery (NEW)
    # ==========================================
    gr.HTML("<br><div class='section-title' style='text-align:center; background:#1e293b; padding:15px; border-radius:12px; border: 1px solid #334155; color: white;'>🗂️ Phase 4: Session Gallery</div>")

    with gr.Row():
        with gr.Column(scale=1, elem_classes="panel-card"):
            gr.HTML("<div class='section-title'>🖼️ All Session Renders</div>")
            with gr.Row():
                gallery_refresh_btn = gr.Button("🔄 Refresh Gallery", variant="secondary")
                gallery_dl_btn = gr.Button("⬇️ Download All as ZIP", variant="primary")

            zip_file_output = gr.File(label="session_gallery.zip", visible=False)

            gallery_view = gr.Gallery(
                label="Generated Images",
                columns=5,
                object_fit="contain",
                height=400
            )


    # --- EVENT TRIGGERS ---

    # 1. ControlNet Generation Logic
    gen_btn.click(
        fn=lambda: "⏳ Generating image... Please wait.", inputs=None, outputs=status_indicator, queue=False
    ).then(
        fn=run_controlnet, inputs=[raw_sketch_input, prompt_input], outputs=[base_output, mask_output]
    ).then(
        fn=lambda: "✅ Generation Complete!", inputs=None, outputs=status_indicator, queue=False
    )

    # 2. Pipeline Routing
    send_to_edit_btn.click(fn=lambda img: img, inputs=[base_output], outputs=[sketchedit_input])
    send_to_inpaint_btn.click(fn=lambda img: img, inputs=[base_output], outputs=[inpaint_input])

    # 3. SketchEdit Execution Logic
    edit_btn.click(
        fn=lambda: "⏳ Synthesizing edits... Please wait.", inputs=None, outputs=edit_status, queue=False
    ).then(
        fn=run_sketchedit_process, inputs=[sketchedit_input], outputs=[sketchedit_output]
    ).then(
        fn=lambda: "✅ Editing Complete!", inputs=None, outputs=edit_status, queue=False
    )

    # 4. Inpaint Execution Logic
    inpaint_btn.click(
        fn=lambda: "⏳ Inpainting region... Please wait.", inputs=None, outputs=inpaint_status, queue=False
    ).then(
        fn=run_inpaint_process, inputs=[inpaint_input, inpaint_prompt], outputs=[inpaint_output]
    ).then(
        fn=lambda: "✅ Inpainting Complete!", inputs=None, outputs=inpaint_status, queue=False
    )

    # 5. Gallery Logic
    gallery_refresh_btn.click(
        fn=get_gallery_images, inputs=None, outputs=[gallery_view]
    )

    gallery_dl_btn.click(
        fn=download_gallery_zip, inputs=None, outputs=[zip_file_output]
    ).then(
        fn=lambda: gr.update(visible=True), inputs=None, outputs=[zip_file_output]
    )

demo.launch(debug=True, share=True, show_api=False)

Cloning into 'sketchedit'...
remote: Enumerating objects: 318, done.
remote: Counting objects: 100% (318/318), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 318 (delta 117), reused 295 (delta 103), pack-reused 0 (from 0)
Receiving objects: 100% (318/318), 28.35 MiB | 25.71 MiB/s, done.
Resolving deltas: 100% (117/117), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.0 MB/s eta 0:00:00
/content/sketchedit
--2026-05-15 17:37:59--  https://maildluteducn-my.sharepoint.com/:u:/g/personal/zengyu_mail_dlut_edu_cn/ETFFjNQcRzFMs2Dgq6tiVnIB3msKBXYMBOSWnVYEHgrlxQ?download=1
Resolving maildluteducn-my.sharepoint.com (maildluteducn-my.sharepoint.com)... 13.107.136.10, 13.107.138.10, 2620:1ec:8f8::10, ...
Connecting to maildluteducn-my.sharepoint.com (maildluteducn-my.sharepoint.com)|13.107.136.10|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: /pers

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Booting Base Generation Models on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0ea2b3ba90488e9c4b.gradio.live
